In [ ]:
from pathlib import Path
import pandas as pd
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork

DATA_URL = (
    "https://raw.githubusercontent.com/plotly/datasets/master/"
    "heart.csv"
)

def load_data() -> pd.DataFrame:
    """Load the public heart dataset and return a compact discrete table."""
    df = pd.read_csv(DATA_URL)
    # Dataset fields: age, sex, cp, trestbps, chol, fbs, restecg, thalach,
    # exang, oldpeak, slope, ca, thal, target.
    df = df[["age", "cp", "thalach", "exang", "target"]].copy()
    # Discretize continuous values so the Bayesian network stays simple.
    df["age"] = pd.cut(
        df["age"], bins=[0, 45, 60, 120], labels=["young", "middle", "older"]
    ).astype(str)
    df["thalach"] = pd.cut(
        df["thalach"], bins=[0, 120, 150, 250], labels=["low", "medium", "high"]
    ).astype(str)
    df["cp"] = df["cp"].astype(str)
    df["exang"] = df["exang"].astype(str)
    df["target"] = df["target"].astype(int).astype(str)
    return df.dropna()

def main() -> None:
    data = load_data()
    print("Dataset rows:", len(data))
    print("Heart disease target counts (0 = absent, 1 = present):")
    print(data["target"].value_counts().sort_index())
    # Edges encode the assumed dependency structure.
    model = DiscreteBayesianNetwork(
        [
            ("age", "thalach"),
            ("age", "target"),
            ("cp", "target"),
            ("thalach", "target"),
            ("exang", "target"),
        ]
    )
    model.fit(data, estimator=MaximumLikelihoodEstimator)
    inference = VariableElimination(model)
    print("\nExample inference: older age group, low max heart rate, exercise angina")
    result = inference.query(
        variables=["target"],
        evidence={"age": "older", "thalach": "low", "exang": "1"},
        show_progress=False,
    )
    print(result)
    print("\nThe probability values are learned from this dataset and model only.")
    # Optional interactive example. Blank input skips evidence for that field.
    print("\nEnter evidence for a second example (press Enter to skip a field):")
    evidence = {}
    choices = {
        "age": ["young", "middle", "older"],
        "thalach": ["low", "medium", "high"],
        "cp": sorted(data["cp"].unique()),
        "exang": sorted(data["exang"].unique()),
    }
    for field, options in choices.items():
        answer = input(f"{field} {options}: ").strip()
        if answer:
            if answer not in options:
                print(f"Ignoring invalid {field} value: {answer}")
            else:
                evidence[field] = answer
    if evidence:
        print("\nModel probability for the selected evidence:")
        print(inference.query(variables=["target"], evidence=evidence, show_progress=False))

if __name__ == "__main__":
    main()